# Two-qutrit Bell vertical slice

This notebook exercises the public `qudits_on_qubits` API from a logical two-qutrit Bell reference through explicit qutrit-to-qubit encoding, qubit-circuit execution on ideal Aer, logical decoding, and integrity-checked manifest loading.

## 1. Import only the public library API

In [ ]:
import os
from pathlib import Path

from qudits_on_qubits import (
    AerIdeal,
    BellReferenceCircuitSpec,
    ExecutionSpec,
    QuditExperimentSpec,
    TranspilationConfig,
    canonical_qutrit_encoding,
    load_run_manifest,
    run_vertical_slice,
)

## 2. Define the logical experiment and explicit encoding

`canonical_ez` maps each logical qutrit to two physical qubits. The fourth computational basis word is marked as leakage. Environment variables let automated tests use a temporary output directory and fewer shots without changing the documented defaults.

In [ ]:
shots = int(os.environ.get("QOQ_NOTEBOOK_SHOTS", "2048"))
if shots < 256:
    raise ValueError("QOQ_NOTEBOOK_SHOTS must be at least 256")
seed = 42
output_root = Path(os.environ.get("QOQ_NOTEBOOK_OUTPUT_ROOT", "artifacts/two_qutrit_bell_notebook"))

logical_circuit = BellReferenceCircuitSpec("two_qutrit")
encoding = canonical_qutrit_encoding()
spec = QuditExperimentSpec(
    circuit=logical_circuit,
    encoding=encoding,
    backend=AerIdeal(seed_simulator=seed),
    execution=ExecutionSpec(
        shots=shots,
        seed=seed,
        transpilation=TranspilationConfig(optimization_level=1, seed_transpiler=seed),
    ),
    output_root=output_root,
    tags={"example": "two-qutrit-bell-notebook"},
)

logical_summary = {
    "reference_id": logical_circuit.reference_id,
    "logical_dimension": encoding.logical_dimension,
    "physical_qubits_per_qutrit": encoding.physical_qubits,
    "encoding_id": encoding.encoding_id,
}
logical_summary

## 3. Inspect the generated qubit circuits

Preparation produces one four-qubit encoded source circuit and nine measured qubit circuits required by the Bell functional.

In [ ]:
prepared = logical_circuit.prepare(encoding)
encoded_circuit_count = len(prepared.executable_circuits)
circuit_summary = {
    "source_circuit_count": len(prepared.source_circuits),
    "source_qubits": prepared.source_circuits[0].num_qubits,
    "encoded_circuit_count": encoded_circuit_count,
    "measured_qubits": prepared.executable_circuits[0].num_qubits,
}
assert circuit_summary == {
    "source_circuit_count": 1,
    "source_qubits": 4,
    "encoded_circuit_count": 9,
    "measured_qubits": 4,
}
circuit_summary

## 4. Execute and decode

The runner compiles and executes the qubit circuits, decodes counts into logical qutrit outcomes, evaluates the Bell functional, and persists linked artifacts.

In [ ]:
completed = run_vertical_slice(spec)
result = dict(completed.result)
tolerance = 0.15 * (2048 / shots) ** 0.5
summary = {
    "status": completed.manifest.status,
    "benchmark": result["benchmark"],
    "encoded_circuit_count": result["circuit_count"],
    "bell_unconditional": result["bell_unconditional"]["real"],
    "bell_conditional": result["bell_conditional"]["real"],
    "leakage_rate": result["leakage_rate"],
}
assert summary["status"] == "completed"
assert summary["benchmark"] == "two_qutrit"
assert summary["encoded_circuit_count"] == 9
assert abs(summary["bell_unconditional"] - 6.0) <= tolerance
assert summary["leakage_rate"] == 0.0
summary

## 5. Reload and verify the run manifest

`load_run_manifest` checks schema, run identity, path containment, file existence, and every artifact SHA-256 before returning the manifest.

In [ ]:
manifest_path = completed.artifact_dir / "run-manifest.json"
manifest = load_run_manifest(completed.artifact_dir)
manifest_summary = {
    "path": str(manifest_path),
    "schema_version": manifest.schema_version,
    "status": manifest.status,
    "execution_mode": manifest.backend.execution_mode.value,
    "artifact_count": len(manifest.artifacts),
}
assert manifest_summary["schema_version"] == "run-manifest-v1"
assert manifest_summary["status"] == "completed"
assert manifest_summary["execution_mode"] == "ideal_simulator"
assert manifest_summary["artifact_count"] == 7
assert manifest_path.is_file()
manifest_summary